# MTG Combined Training — Scorer + Composer

Single notebook that trains both models in one Colab session.

**Estimated time:** ~5 hours on A100 (1hr scorer + 4hr composer)

**Input (Google Drive / MTG-Training /):**
- `scorer_train.jsonl` — card-commander fit rating pairs
- `composer_train.jsonl` — full deck generation sequences

**Output (Google Drive / MTG-Training /):**
- `mtg-scorer-gguf/` — merged scorer model
- `mtg-composer-gguf/` — merged composer model

**v2 improvements:**
- Scorer: color identity violation negatives, popularity-based scoring, more epochs
- Composer: theme-conditioned prompts, more epochs
- Both: disable thinking mode in chat template

## 1. Setup

In [ ]:
!pip install -q unsloth

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/MTG-Training"

import os
SCORER_DATA = f"{DRIVE_DIR}/scorer_train.jsonl"
COMPOSER_DATA = f"{DRIVE_DIR}/composer_train.jsonl"

assert os.path.exists(SCORER_DATA), f"Scorer data not found: {SCORER_DATA}"
assert os.path.exists(COMPOSER_DATA), f"Composer data not found: {COMPOSER_DATA}"

print(f"Scorer data:   {os.path.getsize(SCORER_DATA) / 1e6:.1f} MB")
print(f"Composer data: {os.path.getsize(COMPOSER_DATA) / 1e6:.1f} MB")

---
## 2. Train Scorer

LoRA rank 16, 5 epochs, short context (512 tokens).

In [ ]:
from unsloth import FastLanguageModel

SCORER_SEQ_LEN = 512

scorer_model, scorer_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=SCORER_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

scorer_model = FastLanguageModel.get_peft_model(
    scorer_model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(scorer_model.print_trainable_parameters())

In [ ]:
from datasets import load_dataset

scorer_dataset = load_dataset("json", data_files=SCORER_DATA, split="train")
print(f"Loaded {len(scorer_dataset)} scorer examples")

def format_scorer(examples):
    texts = []
    for messages in examples["messages"]:
        text = scorer_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)
    return {"text": texts}

scorer_dataset = scorer_dataset.map(
    format_scorer, batched=True, remove_columns=scorer_dataset.column_names
)
print("Sample:\n", scorer_dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

scorer_trainer = SFTTrainer(
    model=scorer_model,
    tokenizer=scorer_tokenizer,
    train_dataset=scorer_dataset,
    dataset_text_field="text",
    max_seq_length=SCORER_SEQ_LEN,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        output_dir="mtg-scorer-adapter",
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        num_train_epochs=5,
        bf16=True,
        warmup_ratio=0.05,
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=2,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

scorer_stats = scorer_trainer.train()
print(f"Scorer done. Steps: {scorer_stats.global_step}, Loss: {scorer_stats.training_loss:.4f}")

### Export Scorer

In [ ]:
import shutil

# Save adapter
scorer_model.save_pretrained("mtg-scorer-adapter")
scorer_tokenizer.save_pretrained("mtg-scorer-adapter")

# Merge and save as 16-bit safetensors
scorer_model.save_pretrained_merged(
    "mtg-scorer-merged", scorer_tokenizer, save_method="merged_16bit"
)

# Copy to Drive
drive_scorer = f"{DRIVE_DIR}/mtg-scorer-gguf"
if os.path.exists(drive_scorer):
    shutil.rmtree(drive_scorer)
shutil.copytree("mtg-scorer-merged", drive_scorer)
print(f"Scorer exported to {drive_scorer}")

### Free VRAM

In [ ]:
import torch, gc

del scorer_model, scorer_tokenizer, scorer_trainer, scorer_dataset
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info(0)[0] / 1e9:.1f} GB")

---
## 3. Train Composer

LoRA rank 32, 7 epochs, long context (4096 tokens).

In [ ]:
from unsloth import FastLanguageModel

COMPOSER_SEQ_LEN = 4096

composer_model, composer_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=COMPOSER_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

composer_model = FastLanguageModel.get_peft_model(
    composer_model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(composer_model.print_trainable_parameters())

In [ ]:
from datasets import load_dataset

composer_dataset = load_dataset("json", data_files=COMPOSER_DATA, split="train")
print(f"Loaded {len(composer_dataset)} composer examples")

def format_composer(examples):
    texts = []
    for messages in examples["messages"]:
        text = composer_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)
    return {"text": texts}

composer_dataset = composer_dataset.map(
    format_composer, batched=True, remove_columns=composer_dataset.column_names
)
print("Sample (first 800 chars):\n", composer_dataset[0]["text"][:800])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

composer_trainer = SFTTrainer(
    model=composer_model,
    tokenizer=composer_tokenizer,
    train_dataset=composer_dataset,
    dataset_text_field="text",
    max_seq_length=COMPOSER_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="mtg-composer-adapter",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        num_train_epochs=7,
        bf16=True,
        warmup_ratio=0.05,
        logging_steps=25,
        save_strategy="epoch",
        save_total_limit=2,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

composer_stats = composer_trainer.train()
print(f"Composer done. Steps: {composer_stats.global_step}, Loss: {composer_stats.training_loss:.4f}")

### Export Composer

In [ ]:
import shutil

# Save adapter
composer_model.save_pretrained("mtg-composer-adapter")
composer_tokenizer.save_pretrained("mtg-composer-adapter")

# Merge and save as 16-bit safetensors
composer_model.save_pretrained_merged(
    "mtg-composer-merged", composer_tokenizer, save_method="merged_16bit"
)

# Copy to Drive
drive_composer = f"{DRIVE_DIR}/mtg-composer-gguf"
if os.path.exists(drive_composer):
    shutil.rmtree(drive_composer)
shutil.copytree("mtg-composer-merged", drive_composer)
print(f"Composer exported to {drive_composer}")

---
## 4. Quick Sanity Check

In [ ]:
# Quick test: does the composer output look like a deck?
FastLanguageModel.for_inference(composer_model)

messages = [
    {"role": "system", "content": "You are an expert MTG Commander deckbuilder. Build a 99-card deck for the given commander."},
    {"role": "user", "content": "Build a 99-card Commander deck for Atraxa, Praetors' Voice.\nColors: BGUW\nPower Level (Bracket): 3\nTheme/Strategy: +1/+1 counters and proliferate\n\nList cards grouped by category using '## Category (N)' headers. End with '## Lands (N)' for the land base. One card per line."},
]

input_text = composer_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
    enable_thinking=False,
)
inputs = composer_tokenizer(input_text, return_tensors="pt").to("cuda")

import torch
with torch.no_grad():
    out = composer_model.generate(
        **inputs, max_new_tokens=500, temperature=0.7,
        do_sample=True, repetition_penalty=1.2,
    )
text = composer_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== Composer output (first 500 tokens) ===")
print(text[:1500])

In [ ]:
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\nScorer:   {DRIVE_DIR}/mtg-scorer-gguf/")
print(f"Composer: {DRIVE_DIR}/mtg-composer-gguf/")
print("\nDownload both folders from Google Drive, then place them in:")
print("  Magic/models/Qwen35/mtg-scorer-gguf/")
print("  Magic/models/Qwen35/mtg-composer-gguf/")